# Datamine MIKEST Process Example

This notebook demonstrates how to configure and run the **`mikest`** process wrapper in `dmstudio`.

## Process Description

## MIKEST

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

The MIKEST process uses the Indicator Estimation (IE) method to estimate grades into a block model using the cumulative distribution function (CDF) of indicator transformed sample grades. It is similar to the [INDEST](<indest.md>) process except it is based on the Advanced Estimation [COKRIG](<cokrig.md>) process rather than the [ESTIMA](<estima.md>) process. The main advantage of MIKEST over INDEST is processing speed.

To operate, MIKEST needs a series of threshold values between the smallest and largest grade values. These threshold values, referred to as cutoffs, are used to numerically build the CDF of each block in the model.

For each cutoff, data in the search volume are transformed into zeroes and ones: 1 is assigned if the data are greater than the threshold and 0 if less than or equal to the threshold. MIKEST then estimates the probability that the block grade is greater than the threshold value, using Ordinary Kriging.

Performing this operation for each cutoff across the range of the sample data approximates the CDF for the model cell. After the CDF is built, it is post-processed to calculate the indicator-estimated grade.

MIKEST uses the [COKRIG](<cokrig.md>) process to perform the estimation for each cutoff.

You must already have calculated a variogram for each cutoff and stored the models in the [Variogram Model file](<../COMMON/filetype.md#VariogramMod>). The [VGRAM](<vgram.md>) process has specific features for calculating indicator variograms.

For each cutoff MIKEST calculates the following data which can optionally be stored in the Output Model file:

* PRABi: the proportion (fraction) of the model subcell which is above cutoff number i
* GRABi: the grade of the proportion of the subcell which lies above cutoff.

The main output from MIKEST is the grade above a cutoff of zero, i.e. the indicator-estimated grade of the total subcell.

### Input Files:

* **proto** (Block_Model_File):
  Input model prototype This is a standard block model file containing the 13 compulsory
  fields. It may also contain the rotated model fields. If it includes cells then it must
  be sorted on IJK.
  Required=Yes

* **samples** (Undefined):
  Input sample data This must contain X,Y and Z fields and at least one grade field.
  Required=Yes

* **spar** (Undefined):
  Search volume parameter file The file contains 13 required fields as described below.
  For optional fields refer to the COKRIG online Help . Required Fields:
  Required=Yes

* **epar** (Undefined):
  Estimation parameter file. Each record in the file describes an estimation method and
  its associated numeric parameters. There are 9 required fields as described below.
  Required=Yes

* **fields** (Undefined):
  Estimation Fields file. This file is used to define field names associated with each
  EREFNUM defined in the EPAR file. There are two required fields as described below:
  Required=Yes

* **vmodel** (Variogram - Model):
  Variogram model parameter file. Each record in this file defines a variogram model type
  and its parameters. There are 13 required fields as described below.
  Required=Yes

### Output Files:

* **outmodel** (Block Model File):
  Output model containing estimated MIK grades, etc.
  Required=No

* **avgrades** (Undefined):
  Output file containing cutoff grade ranges and average grade used for each range. It
  will include zone field(s), if any, plus the following fields:
  Required=No

* **indicate** (Undefined):
  Output indicator file. This is a copy of the sample input SAMPLES file, but also
  includes the 0/1 indicator values for each cutoff
  Required=No

* **sampout** (Undefined):
  Output sample file containing details of weights for each sample for each cell
  estimated.
  Required=No

### Fields:

### Parameters:

* **order**:
  Order relation correction method:
  Range=
  Values=
  Default=
  Required=No

* **grmethod**:
  Method for specifying average grade within each cutoff range:
  Range=
  Values=
  Default=
  Required=No

* **pgfields**:
  Flag to select whether the proportion above cutoff fields (PRABn) and the grade above
  cutoff fields (GRABn) should be included in the OUTMODEL file:
  Range=
  Values=
  Default=
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('mikest')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute mikest
print("Running mikest...")
dm_cmd.mikest(
    proto_i='t_mod1',  # required
    samples_i=['t_assays'],  # required
    spar_i='t_input_file',  # required
    epar_i='t_input_file',  # required
    fields_i=['t_input_file'],  # required
    vmodel_i='t_mod1',  # required
    # outmodel_o='t_mikest_out',  # optional
    # avgrades_o=['t_mikest_out'],  # optional
    # indicate_o='t_mikest_out',  # optional
    # sampout_o='t_mikest_out',  # optional
    # order_p='optional',  # optional
    # grmethod_p='optional',  # optional
    # pgfields_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("mikest execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_mikest_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")